# Common Libraries

In [1]:
import os, sys, optax
import numpy as np
import pandas as pd
import equinox as eqx
from tqdm import tqdm, trange
import matplotlib.pylab as plt
from typing import Tuple, Dict

"""
Setting for GPU acceleration
"""
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false" # limit jax and TF from consuming all GPU memory

# Jax
import jax
jax.config.update("jax_enable_x64", True)
from jax import numpy as jnp
from jaxtyping import Float, Array

# Mujoco
import mujoco

# Custom Libraries

In [2]:
sys.path.append("/home/seojin/Tools/biomechanics/MonocularBiomechanics")
from monocular_demos.biomechanics_mjx.forward_kinematics import ForwardKinetics
from monocular_demos.biomechanics_mjx.monocular_trajectory import KineticsWrapper

In [3]:
# construct a loss function between the forward pass through the forward kinematic
# implicit representation and the resulting keypoint and the detected keypoitns
def loss(
    model: KineticsWrapper,
    x: Float[Array, "times"],
    y: Float[Array, "times keypoints 3"],
    target_site_idx,
    site_offset_regularization = 1e-1
) -> Tuple[Float, Dict]:

    timestamps = x
    keypoints3d = y # unit: m
    metrics = {}

    # NOTE: steps is an make sure this retraces for different dimensions
    (state, constraints, next_states), (ang, vel, action), _ = model(
        timestamps,
        skip_vel=True,
        skip_action=True,
    )

    pred_kp3d = state.site_xpos[:, target_site_idx, :] # unit: m, shape: (#datas, #sites, 3)
    l = jnp.mean((pred_kp3d - keypoints3d) ** 2) * 100 # so in cm
    metrics["kp_err"] = l

    # regularize marker offset
    l_site_offset = jnp.sum(jnp.square(model.site_offsets))
    l += l_site_offset * site_offset_regularization

    # make loss the first key in the dictionary by popping and building a new dictionary with the rest
    metrics = {"loss": l, **metrics}

    return l, metrics

@eqx.filter_jit
def step(model, opt_state, data, loss_grad, optimizer, **kwargs):
    x, targets = data

    (val, metrics), grads = loss_grad(model, x=x, y=targets, **kwargs)
    updates, opt_state = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_inexact_array))
    model = eqx.apply_updates(model, updates) # Update model with preserving immutable property
    
    return val, model, opt_state, metrics

# Params

In [4]:
model_path = "/mnt/sdb2/DeepProprioception/MyoSuite/muscle_mimic/muscle_mimic_scaled.xml"
pose3d_path = "/mnt/ext1/seojin/temp/DP/Rendering/output/smplv2/smpl_pose_run1.npz"

# Constants

In [5]:
# SMPL <-> Muscle Mimic body name
mapping_smpl_mm = {
    "R_Shoulder" : "humerus_r",
    "R_Elbow" : "ulna_r",
    "R_Wrist" : "lunate_r",
}

# Make model

In [6]:
fk = ForwardKinetics(xml_path = model_path)

actuator_names = np.array([mujoco.mj_id2name(fk.model, mujoco.mjtObj.mjOBJ_ACTUATOR, i) for i in range(fk.model.nu)])

body_names = np.array([mujoco.mj_id2name(fk.model, mujoco.mjtObj.mjOBJ_BODY, i) for i in range(fk.model.nbody)])
excludes = ["world", "Bimanual", "root"]
body_segment_names = body_names[np.isin(body_names, excludes) == False]

site_names = np.array([mujoco.mj_id2name(fk.model, mujoco.mjtObj.mjOBJ_SITE, i) for i in range(fk.model.nsite)])

mimic_site_idx = jnp.array([fk.site_names.index(f"{name}_Mimic_Site") for name in mapping_smpl_mm])

# Make scale mix

Each key in mappings represents a segment group, and the corresponding list contains the subsegments belonging to that group

In [7]:
# Scale mix
mappings = {
    "torso" : ["thorax", "cervical_spine"],
    "shoulder_r" : ["clavicle_r", "scapula_r"],
    "upperarm_r" : ["humerus_r"],
    "forearm_r" : ["radius_r", "ulna_r"],
    "overall" : body_segment_names,
}

scale_mixer = jnp.array([
    [1.0 if body in child_segments else 0.0 
     for segment, child_segments in mappings.items()]
    for body in body_names
])

In [8]:
"""
KineticsWrapper consists of a neural network and a forward kinematics module, 
    where the scaling factor for each body segment is computed from body_scale and scale_mixer

Property
    - NonLearnable
        - scale_mixer

    - Learnable
        - body_scale
        - site_offsets: offset of each site
"""
model = KineticsWrapper(fk, scale_mixer = scale_mixer)

/tmp/ipykernel_510364/1986745145.py:13: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  model = KineticsWrapper(fk, scale_mixer = scale_mixer)


# Load data & Preprocessing

In [9]:
# Load pose
smpl_data = np.load(pose3d_path)
pose3d = smpl_data["kp3ds"]
kp_labels = list(smpl_data["kp_label"])
pose3d = pose3d[:10, :, :]

# Filtering
target_smpl_kp_idx = [ kp_labels.index(smpl_kp) for smpl_kp in mapping_smpl_mm ]
pose3d = pose3d[:, target_smpl_kp_idx, :]

# Make dataset
timestamps = jnp.arange(len(pose3d)) / 30.0
dataset = (timestamps, pose3d)

# Calculate loss

In [10]:
results = model(timestamps, skip_vel = True, skip_action = True)

kinetics_output = results[0]
trajectory_output = results[1]

state = kinetics_output[0]
constraints = kinetics_output[1]
next_states = kinetics_output[2]

joint_angle = trajectory_output[0]
joint_velocity = trajectory_output[1]
action = trajectory_output[2]

In [11]:
pred_kp3d = state.site_xpos[:, mimic_site_idx, :]
correct_kp3d = dataset[1]

In [12]:
site_offset_regularization = 1e-1

In [13]:
l = jnp.mean((pred_kp3d - correct_kp3d) ** 2) * 100 # so in cm

metrics = {}
metrics["kp_err"] = l

# regularize marker offset
l_site_offset = jnp.sum(jnp.square(model.site_offsets))
l += l_site_offset * site_offset_regularization

In [14]:
# make loss the first key in the dictionary by popping and building a new dictionary with the rest
metrics = {"loss": l, **metrics}

# Fit model

In [15]:
lr_end_value = 1e-8
lr_init_value = 1e-4
max_iters: int = 5000
clip_by_global_norm = 0.1

In [16]:
# work out the transition steps to make the desired schedule
transition_steps = 10
lr_decay_rate = (lr_end_value / lr_init_value) ** (1.0 / (max_iters // transition_steps))
learning_rate = optax.warmup_exponential_decay_schedule(
    init_value=0,
    warmup_steps=0,
    peak_value=lr_init_value,
    end_value=lr_end_value,
    decay_rate=lr_decay_rate,
    transition_steps=transition_steps,
)

In [17]:
optimizer = optax.chain(
    optax.adamw(learning_rate=learning_rate, b1=0.8, weight_decay=1e-5),
    optax.zero_nans(),
    optax.clip_by_global_norm(clip_by_global_norm)
)

In [18]:
opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))

In [19]:
loss_grad = eqx.filter_value_and_grad(loss, has_aux=True)

In [ ]:
counter = trange(max_iters)
for i in counter:
    # val, model, opt_state, metrics = step(model, opt_state, dataset, loss_grad, optimizer)
    
    x, targets = dataset
    (loss, metrics), gradients = loss_grad(model, x=x, y=targets, target_site_idx = mimic_site_idx) # metrics: auxiliary data
    updates, opt_state = optimizer.update(gradients, opt_state, eqx.filter(model, eqx.is_inexact_array))
    model = eqx.apply_updates(model, updates)

    # Print optimization result
    if i > 0 and i % int(max_iters // 10) == 0:
        print(f"iter: {i} loss: {loss}.")  # metrics: {metrics}")

    if i % 50 == 0:
        metrics = {k: v.item() for k,v in metrics.items()}
        print(loss, metrics)

  0%|          | 1/5000 [00:23<33:01:17, 23.78s/it]

823.9043088119381 {'kp_err': 823.9043088106348, 'loss': 823.9043088119381}


  1%|          | 51/5000 [00:41<27:39,  2.98it/s]  

73.18303613004562 {'kp_err': 73.18303127001093, 'loss': 73.18303613004562}


  2%|▏         | 101/5000 [01:01<28:56,  2.82it/s]

...

 64%|██████▍   | 3201/5000 [23:36<13:19,  2.25it/s]

32.48704698298356 {'kp_err': 32.48640525759109, 'loss': 32.48704698298356}


 65%|██████▌   | 3251/5000 [24:00<12:30,  2.33it/s]

32.47857148775162 {'kp_err': 32.47792939951175, 'loss': 32.47857148775162}


 66%|██████▌   | 3295/5000 [24:21<12:15,  2.32it/s]

# Visualization

In [86]:
body_scale_vals = np.array(model.body_scale).flatten()
final_body_scales = (1.0 + np.array(model.scale_mixer) @ body_scale_vals).flatten()
body_names = ["world"] + list(model.forward.body_names)
scale_df = pd.DataFrame({
    "Segment": body_names,
    "Scale_Factor": final_body_scales,
})

scale_df

,Segment,Scale_Factor
0,world,1.000000
1,Bimanual,1.000000
2,root,1.000000
3,thorax,0.981886
4,cervical_spine,0.981886
5,clavicle_r,1.000299
6,clavphant_r,1.007672
7,scapula_r,1.000299
8,scapphant_r,1.007672
9,humphant_r,1.007672


In [87]:
offsets_mm = np.array(model.site_offsets[mimic_site_idx]) * 1000.0   # m -> mm 변환
total_shift_mm = np.linalg.norm(offsets_mm, axis=-1)

site_names = np.array(model.forward.site_names)[mimic_site_idx]

offset_df = pd.DataFrame({
    "Site_Name": site_names,
    "Offset_X (mm)": offsets_mm[:, 0],
    "Offset_Y (mm)": offsets_mm[:, 1],
    "Offset_Z (mm)": offsets_mm[:, 2],
    "Total_Shift (mm)": total_shift_mm
})

print("\n=== [2] 사이트(마커)별 오프셋 이동량 (이동량 큰 순서) ===")
display(offset_df.sort_values(by="Total_Shift (mm)", ascending=False))


=== [2] 사이트(마커)별 오프셋 이동량 (이동량 큰 순서) ===


,Site_Name,Offset_X (mm),Offset_Y (mm),Offset_Z (mm),Total_Shift (mm)
0,R_Shoulder_Mimic_Site,-53.036740,-51.562929,-9.767340,74.612548
1,R_Elbow_Mimic_Site,-0.370890,-29.225436,-2.133648,29.305564
2,R_Wrist_Mimic_Site,0.467559,-0.519699,-0.099152,0.706066
